In [1]:
import networkx as nx
import csv
import matplotlib.pyplot as plt
import pandas as pd
import torch
import numpy as np
import collections
from datetime import datetime
from torch_geometric.data import HeteroData
from collections import Counter, defaultdict
from networkx import degree_centrality, closeness_centrality, betweenness_centrality
from networkx import eigenvector_centrality, clustering
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
def load_raw_data():
    df = pd.read_csv('Dataset/fundamental.csv')
    return df

def calculate_financial_ratios(dataset):
    df = dataset.copy()
    cols_to_numeric = ['epspx', 'prcc_f', 'ceq', 'csho', 'revt', 'cogs', 'at', 'ni', 'che', 'ap']
    df[cols_to_numeric] = df[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

    df['eps'] = df['epspx']
    df['pe_ratio'] = df['prcc_f'] / df['epspx']
    df['pb_ratio'] = df['prcc_f'] / (df['ceq'] / df['csho'])
    df['gross_margin'] = (df['revt'] - df['cogs']) / df['revt']
    df['asset_turnover'] = df['revt'] / df['at']
    df['roe'] = df['ni'] / df['ceq']
    df['roa'] = df['ni'] / df['at']
    df['invested_capital_2'] = df['at'] - df['che'] - df['ap']
    df['roi'] = df['ni'] / df['invested_capital_2']
    df['fyear'] = df['fyear'].astype('Int64')

    df = df.sort_values(by=['gvkey', 'fyear'])

    selected_cols = ['gvkey', 'fyear', 'eps', 'pe_ratio', 'pb_ratio', 'gross_margin', 'cogs', 'asset_turnover', 'chech', 'che', 'ni', 'roe', 'roi', 'revt', 'roa']
    
    df_new = df[selected_cols].copy()
    
    return df_new 

def preprocess_data():
    dataset = load_raw_data()
 
    # dataset = map_gvkey_companyid(dataset)

    dataset = calculate_financial_ratios(dataset)

    return dataset

In [3]:
def load_graph(year):
  postfix = ''
  # if year == 2022:
  #   postfix = '_test'

  GSC = nx.read_gml(f'Dataset/SCNs/NW_{year}_stock_comps_v1_connect{postfix}.gml')
  GB = nx.read_gml(f"Dataset/BoardexNetwork/NW_{year-1}_boardex.gml")
  GC = nx.read_gml(f"Dataset/CN/NW_{year-1}_competition.gml")

  return GSC, GB, GC

def edge_index_from_graph(G, node_to_idx):
  edges = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges() if u in node_to_idx and v in node_to_idx]
  return torch.tensor(edges, dtype=torch.long).t().contiguous()

def compute_struct_features(G, name):
        struct_feats = {}
        centrality_funcs = {
            f'{name}_deg': degree_centrality,
            f'{name}_clo': closeness_centrality,
            f'{name}_bet': betweenness_centrality,
            f'{name}_eig': eigenvector_centrality,
            f'{name}_clu': clustering
        }
        for k, func in centrality_funcs.items():
            try:
                result = func(G)
                for node, val in result.items():
                    if node not in struct_feats:
                        struct_feats[node] = {}
                    struct_feats[node][k] = val
            except Exception as e:
                print(f"Failed to compute {k}: {e}")
        return struct_feats

def get_features_year(year, dataset, all_nodes, common_nodes):
    original_features = ['mergent_id', 'eps', 'pe_ratio', 'pb_ratio', 'gross_margin', 'cogs', 'asset_turnover', 'chech', 'che', 'ni', 'roe', 'roi', 'revt', 'roa']
    features = ['mergent_id', 'eps', 'pe_ratio', 'pb_ratio', 'cogs', 'asset_turnover', 'chech', 'che', 'ni', 'roe', 'revt', 'roa']
    
    temp_df = dataset.loc[(dataset['fyear'] == year - 1), features]
    
    curr_labels_df = dataset.loc[dataset['fyear'] == year, ['mergent_id', 'revt', 'roa']]
    curr_labels_df.rename(columns={'revt': 'curr_revt', 'roa': 'curr_roa'}, inplace=True)
    temp_df = temp_df.merge(curr_labels_df, on='mergent_id', how='left')

    temp_df['growth_revt'] = (temp_df['curr_revt'] - temp_df['revt']) / temp_df['revt']
    temp_df['growth_roa'] = (temp_df['curr_roa'] - temp_df['roa']) / temp_df['roa']

    # print(temp_df.shape)
    temp_df = temp_df.dropna()

    numeric_cols = temp_df.select_dtypes(include=[np.number]).columns
    temp_df = temp_df[~temp_df[numeric_cols].isin([np.inf, -np.inf]).any(axis=1)]

    if np.isinf(temp_df.select_dtypes(include=[np.number])).any().any():
        print("Warning: DataFrame contains infinite values.")
        print((temp_df.select_dtypes(include=[np.number]) == np.inf).sum())
    
    temp_labels_df = temp_df[(temp_df['mergent_id'].isin(common_nodes))]
    median_revt = temp_labels_df['growth_revt'].median()
    median_roa = temp_labels_df['growth_revt'].median()

    temp_labels_df['label_revt'] = np.where(temp_labels_df['growth_revt']> median_revt, 1, 0)
    temp_labels_df['label_roa'] = np.where(temp_labels_df['growth_revt'] > median_roa, 1, 0)

    features_df = temp_df.loc[(temp_df['mergent_id'].isin(all_nodes)), features]
    labels_df = temp_labels_df[['mergent_id', 'label_revt']]

    print(len(common_nodes))
    print(len(features_df))
    print(len(labels_df))
    
    scaler = StandardScaler()
    features_scaled = features_df.copy()
    features_scaled.iloc[:, 1:] = scaler.fit_transform(features_df.iloc[:, 1:])
    
    return features_scaled, labels_df

In [4]:
def map_gvkey_companyid(dataset, all_nodes, year):
    mapping_file = pd.read_csv('Dataset/mergent2boardexmap.csv')
    nodes_df = pd.DataFrame({'mergent_id': list(all_nodes)})

    nodes_df['mergent_id'] = nodes_df['mergent_id'].astype(str)
    mapping_file['mergent_id'] = mapping_file['mergent_id'].astype(str)

    mapping_file['First Effective Date of Link'] = pd.to_datetime(mapping_file['First Effective Date of Link'], errors='coerce')
    mapping_file['Last Effective Date of Link'] = pd.to_datetime(mapping_file['Last Effective Date of Link'], errors='coerce')
    start = pd.Timestamp(f'{year}-05-01', tz='UTC')
    end = pd.Timestamp(f'{year}-05-15', tz='UTC')

    filtered = mapping_file[(mapping_file['First Effective Date of Link'] <= end) & (mapping_file['Last Effective Date of Link'].isna() | (mapping_file['Last Effective Date of Link'] >= start))]
    
    nodes_mapped = pd.merge(nodes_df, filtered[['mergent_id','GVKEY']], on='mergent_id', how='inner')
    nodes_mapped = nodes_mapped.drop_duplicates()

    # if '75437' in nodes_mapped['mergent_id'].values:
    #     print(nodes_mapped[nodes_mapped['mergent_id'] == '75437'])
    
    mapping_dict = dict(zip(nodes_mapped['GVKEY'].astype(str).str.lstrip('0'), nodes_mapped['mergent_id']))
 
    dataset['mergent_id'] = dataset['gvkey'].astype(str).map(mapping_dict)

    dataset = dataset.dropna(subset=['mergent_id']).copy()
    dataset.loc[:, 'mergent_id'] = dataset['mergent_id'].astype(int).astype(str)
    
    return dataset

In [5]:
def get_data(year, dataset): 
    GSC, GB, GC = load_graph(year)
    nodes_supply = set(GSC.nodes())
    nodes_boardex = set(GB.nodes())
    nodes_compete = set(GC.nodes())

    all_nodes = sorted(list(nodes_supply | nodes_boardex | nodes_compete))
    common_nodes = sorted(list(nodes_supply & nodes_boardex & nodes_compete))
    node_to_idx = {node: i for i, node in enumerate(all_nodes)}

    dataset = map_gvkey_companyid(dataset, all_nodes, year)
    features_df, labels_df = get_features_year(year, dataset, all_nodes, common_nodes)

    features_df['mergent_id'] = features_df['mergent_id'].astype(str)
    feat_dict = features_df.set_index('mergent_id').to_dict('index')
    labels_df['mergent_id'] = labels_df['mergent_id'].astype(str)
    label_dict = dict(zip(labels_df['mergent_id'], labels_df[f'label_revt']))

    print(f"# of Nodes with Features:{len(feat_dict)}")
    print(f"# of Nodes with Labels:{len(label_dict)}")
    print(f"# of Nodes in SCN:{len(nodes_supply)}")
    print(f"# of Nodes in Boardex:{len(nodes_boardex)}")
    print(f"# of Nodes in Competition:{len(nodes_compete)}")
    print(f"# of Nodes in All:{len(all_nodes)}")
    print(f"# of Nodes in Common:{len(common_nodes)}")

    num_features = features_df.shape[1] - 1
    x = torch.zeros((len(all_nodes), num_features), dtype=torch.float)
    y = torch.full((len(all_nodes),), -1, dtype=torch.long)

    for node in all_nodes:
        idx = node_to_idx[node]
        if node in feat_dict:
            x[idx] = torch.tensor(list(feat_dict[node].values()), dtype=torch.float)

        if node in label_dict:
            y[idx] = torch.tensor(label_dict[node], dtype=torch.long)

    struct_feats = defaultdict(dict)
    for graph, name in zip([GSC, GB, GC], ['sup', 'brd', 'com']):
        local_feats = compute_struct_features(graph, name)
        for node in local_feats:
            struct_feats[node].update(local_feats[node])

    all_struct_feats = sorted(next(iter(struct_feats.values())).keys())
    struct_x = torch.zeros((len(all_nodes), len(all_struct_feats)), dtype=torch.float)
    for node in common_nodes:
        idx = node_to_idx[node]
        for i, feat in enumerate(all_struct_feats):
            struct_x[idx, i] = struct_feats[node].get(feat, 0.0)
            
    data = HeteroData()
    data['company'].x = x
    data['company'].struct_x = struct_x
    data['company'].y = y

    graphs = {'supply': GSC, 'boardex': GB, 'competition': GC}

    edge_attrs_list = []
    edge_ranges = {}
    start_idx = 0

    for rel, G in graphs.items():
        edge_index = edge_index_from_graph(G, node_to_idx)
        data[('company', rel, 'company')].edge_index = edge_index
        
        rel_one_hot = {
            'supply': [1, 0, 0],
            'boardex': [0, 1, 0],
            'competition': [0, 0, 1]
        }
        edge_attr = torch.tensor([rel_one_hot[rel]] * edge_index.size(1), dtype=torch.float)
        data[('company', rel, 'company')].edge_attr = edge_attr

        edge_attrs_list.append(edge_attr)
        edge_ranges[rel] = (start_idx, start_idx + edge_index.size(1))
        start_idx += edge_index.size(1)

    data['edges'].edge_x = torch.cat(edge_attrs_list, dim=0)
    data['edges'].edge_ranges = edge_ranges

    for rel, G in graphs.items():
        edge_index = data[('company', rel, 'company')].edge_index
        num_edges = edge_index.size(1)

        node_to_eids = defaultdict(list)
        for eid in range(num_edges):
            src = edge_index[0, eid].item()
            dst = edge_index[1, eid].item()
         
            if all_nodes[src] in common_nodes and all_nodes[dst] in common_nodes:
                node_to_eids[src].append(eid)
                node_to_eids[dst].append(eid)

        edge_pairs = set()
        for eids in node_to_eids.values():
            for i in range(len(eids)):
                for j in range(i + 1, len(eids)):
                    edge_pair = (min(eids[i], eids[j]), max(eids[i], eids[j]))
                    edge_pairs.add(edge_pair)

        if edge_pairs:
            edge_edge_index = torch.tensor(list(edge_pairs), dtype=torch.long).T  # shape [2, num_edges]
        else:
            edge_edge_index = torch.empty((2, 0), dtype=torch.long)

        data[(rel, 'company', rel)].edge_edge_index = edge_edge_index

    num_nodes = len(all_nodes)
    self_edge_index = torch.arange(num_nodes, dtype=torch.long).unsqueeze(0).repeat(2, 1)
    data['company', 'self', 'company'].edge_index = self_edge_index

    common_idx = [node_to_idx[n] for n in common_nodes if n in feat_dict and n in label_dict]
    labels_for_strat = [label_dict[all_nodes[i]] for i in common_idx]
    train_idx, temp_idx, y_train, y_temp = train_test_split(common_idx, labels_for_strat, test_size=0.3, random_state=42, stratify=labels_for_strat)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=y_temp)

    train_mask = torch.zeros(len(all_nodes), dtype=torch.bool)
    val_mask = torch.zeros(len(all_nodes), dtype=torch.bool)
    test_mask = torch.zeros(len(all_nodes), dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    data['company'].train_mask = train_mask
    data['company'].val_mask = val_mask
    data['company'].test_mask = test_mask

    print("Train label distribution:", Counter(y[train_mask].cpu().numpy()))
    print("Val label distribution:", Counter(y[val_mask].cpu().numpy()))
    print("Test label distribution:", Counter(y[test_mask].cpu().numpy()))

    return data

In [6]:
def main():
    years = [2019, 2020, 2021, 2022]
    dataset = preprocess_data()
    for year in years:
        data = get_data(year, dataset)
        torch.save(data, f'Dataset/heterographdata{year}2.pt')
        print(data)
        

In [7]:
if __name__ == "__main__":
    main()

/tmp/ipykernel_10869/3083667385.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_revt'] = np.where(temp_labels_df['growth_revt']> median_revt, 1, 0)
/tmp/ipykernel_10869/3083667385.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_roa'] = np.where(temp_labels_df['growth_revt'] > median_roa, 1, 0)


1911
2288
1774
# of Nodes with Features:2288
# of Nodes with Labels:1774
# of Nodes in SCN:2537
# of Nodes in Boardex:2243
# of Nodes in Competition:2205
# of Nodes in All:2537
# of Nodes in Common:1911
Train label distribution: Counter({1: 621, 0: 620})
Val label distribution: Counter({1: 133, 0: 133})
Test label distribution: Counter({0: 134, 1: 133})
HeteroData(
  company={
    x=[2537, 11],
    struct_x=[2537, 15],
    y=[2537],
    train_mask=[2537],
    val_mask=[2537],
    test_mask=[2537],
  },
  edges={
    edge_x=[37305, 3],
    edge_ranges={
      supply=[2],
      boardex=[2],
      competition=[2],
    },
  },
  (company, supply, company)={
    edge_index=[2, 10256],
    edge_attr=[10256, 3],
  },
  (company, boardex, company)={
    edge_index=[2, 6458],
    edge_attr=[6458, 3],
  },
  (company, competition, company)={
    edge_index=[2, 20591],
    edge_attr=[20591, 3],
  },
  (supply, company, supply)={ edge_edge_index=[2, 135778] },
  (boardex, company, boardex)={ edge_

/tmp/ipykernel_10869/3083667385.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_revt'] = np.where(temp_labels_df['growth_revt']> median_revt, 1, 0)
/tmp/ipykernel_10869/3083667385.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_roa'] = np.where(temp_labels_df['growth_revt'] > median_roa, 1, 0)


1815
2221
1703
# of Nodes with Features:2221
# of Nodes with Labels:1703
# of Nodes in SCN:2420
# of Nodes in Boardex:2135
# of Nodes in Competition:2100
# of Nodes in All:2420
# of Nodes in Common:1815
Train label distribution: Counter({1: 596, 0: 596})
Val label distribution: Counter({0: 128, 1: 127})
Test label distribution: Counter({1: 128, 0: 128})
HeteroData(
  company={
    x=[2420, 11],
    struct_x=[2420, 15],
    y=[2420],
    train_mask=[2420],
    val_mask=[2420],
    test_mask=[2420],
  },
  edges={
    edge_x=[34003, 3],
    edge_ranges={
      supply=[2],
      boardex=[2],
      competition=[2],
    },
  },
  (company, supply, company)={
    edge_index=[2, 11030],
    edge_attr=[11030, 3],
  },
  (company, boardex, company)={
    edge_index=[2, 6151],
    edge_attr=[6151, 3],
  },
  (company, competition, company)={
    edge_index=[2, 16822],
    edge_attr=[16822, 3],
  },
  (supply, company, supply)={ edge_edge_index=[2, 209861] },
  (boardex, company, boardex)={ edge_

/tmp/ipykernel_10869/3083667385.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_revt'] = np.where(temp_labels_df['growth_revt']> median_revt, 1, 0)
/tmp/ipykernel_10869/3083667385.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_roa'] = np.where(temp_labels_df['growth_revt'] > median_roa, 1, 0)


1683
2090
1575
# of Nodes with Features:2090
# of Nodes with Labels:1575
# of Nodes in SCN:2271
# of Nodes in Boardex:2012
# of Nodes in Competition:1942
# of Nodes in All:2271
# of Nodes in Common:1683
Train label distribution: Counter({0: 551, 1: 551})
Val label distribution: Counter({0: 118, 1: 118})
Test label distribution: Counter({0: 119, 1: 118})
HeteroData(
  company={
    x=[2271, 11],
    struct_x=[2271, 15],
    y=[2271],
    train_mask=[2271],
    val_mask=[2271],
    test_mask=[2271],
  },
  edges={
    edge_x=[27596, 3],
    edge_ranges={
      supply=[2],
      boardex=[2],
      competition=[2],
    },
  },
  (company, supply, company)={
    edge_index=[2, 9777],
    edge_attr=[9777, 3],
  },
  (company, boardex, company)={
    edge_index=[2, 5317],
    edge_attr=[5317, 3],
  },
  (company, competition, company)={
    edge_index=[2, 12502],
    edge_attr=[12502, 3],
  },
  (supply, company, supply)={ edge_edge_index=[2, 142774] },
  (boardex, company, boardex)={ edge_ed

/tmp/ipykernel_10869/3083667385.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_revt'] = np.where(temp_labels_df['growth_revt']> median_revt, 1, 0)
/tmp/ipykernel_10869/3083667385.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_labels_df['label_roa'] = np.where(temp_labels_df['growth_revt'] > median_roa, 1, 0)


1552
1958
1452
# of Nodes with Features:1958
# of Nodes with Labels:1452
# of Nodes in SCN:2131
# of Nodes in Boardex:1859
# of Nodes in Competition:1824
# of Nodes in All:2131
# of Nodes in Common:1552
Train label distribution: Counter({0: 508, 1: 508})
Val label distribution: Counter({0: 109, 1: 109})
Test label distribution: Counter({1: 109, 0: 109})
HeteroData(
  company={
    x=[2131, 11],
    struct_x=[2131, 15],
    y=[2131],
    train_mask=[2131],
    val_mask=[2131],
    test_mask=[2131],
  },
  edges={
    edge_x=[25597, 3],
    edge_ranges={
      supply=[2],
      boardex=[2],
      competition=[2],
    },
  },
  (company, supply, company)={
    edge_index=[2, 8775],
    edge_attr=[8775, 3],
  },
  (company, boardex, company)={
    edge_index=[2, 4455],
    edge_attr=[4455, 3],
  },
  (company, competition, company)={
    edge_index=[2, 12367],
    edge_attr=[12367, 3],
  },
  (supply, company, supply)={ edge_edge_index=[2, 98542] },
  (boardex, company, boardex)={ edge_edg